In [1]:
from openai import OpenAI

client = OpenAI()

Distillation → Train a smaller model to mimic a larger one.

In [2]:
# Step 1: Teacher model generates "soft labels"
teacher_response = client.chat.completions.create(
    model="gpt-4o-mini",   # larger, more capable model
    messages=[{"role": "user", "content": "Summarize the importance of cloud computing."}],
    temperature=0.7
)

teacher_output = teacher_response.choices[0].message.content
print("Teacher Output:", teacher_output)

Teacher Output: Cloud computing is crucial for several reasons:

1. **Scalability**: It allows businesses to scale resources up or down based on demand, enabling flexibility to adapt to changing needs.

2. **Cost Efficiency**: Organizations can reduce capital expenses by utilizing pay-as-you-go models, minimizing the need for physical infrastructure and maintenance.

3. **Accessibility**: Cloud services can be accessed from anywhere with an internet connection, facilitating remote work and collaboration across geographically dispersed teams.

4. **Disaster Recovery and Backup**: Cloud computing offers robust data backup and recovery solutions, enhancing data security and reducing the risk of data loss.

5. **Innovation and Agility**: It enables rapid deployment of new applications and services, fostering innovation and allowing businesses to respond quickly to market changes.

6. **Collaboration**: Cloud platforms promote enhanced collaboration among users, allowing real-time updates a

In [3]:
# Step 2: Student model learns from teacher output
student_response = client.chat.completions.create(
    model="gpt-3.5-turbo",   # smaller, distilled model
    messages=[{"role": "user", "content": f"Learn from this teacher summary: {teacher_output}"}],
    temperature=0.7
)

print("Student Output:", student_response.choices[0].message.content)

Student Output: Cloud computing is a critical component for businesses looking to scale their resources, reduce costs, enhance accessibility, improve disaster recovery and backup solutions, drive innovation and agility, foster collaboration, and lessen their environmental impact. By harnessing the power of the cloud, organizations can streamline their operations, stay competitive, and adapt to the ever-changing technological landscape.


Quantization → Compress the same model by reducing precision.

In [4]:
# Example: Using a quantized model (conceptual)
# In practice, quantization is applied during deployment (e.g., INT8 weights).
# Here we just show how you'd call a lighter model that has been quantized.

quantized_response = client.chat.completions.create(
    model="gpt-4o-mini",   # assume this is deployed in INT8 quantized form
    messages=[{"role": "user", "content": "Explain neural networks in simple terms."}],
    temperature=0
)

print("Quantized Model Output:", quantized_response.choices[0].message.content)

Quantized Model Output: Sure! Think of a neural network as a system that tries to mimic how our brains work to solve problems or make decisions.

1. **Basic Structure**: A neural network is made up of layers of nodes, which are like tiny processing units. The first layer is called the input layer, where data enters the network. The last layer is the output layer, where the result comes out. Between these two, there can be one or more hidden layers that help process the information.

2. **Nodes and Connections**: Each node in a layer is connected to nodes in the next layer. These connections have weights, which are like adjustable knobs that help the network learn. When data passes through the network, it gets multiplied by these weights.

3. **Activation Function**: After the data is processed in a node, it goes through an activation function, which decides whether the node should "fire" (send a signal to the next layer) or not. This helps the network make decisions based on the input 

In [5]:
from openai import OpenAI

client = OpenAI()

# Teacher model (larger, more capable)
teacher_model = "gpt-4o-mini"
# Student model (smaller, distilled)
student_model = "gpt-3.5-turbo"

training_data = [
    "Explain cloud computing.",
    "What is reinforcement learning?",
    "Summarize the importance of cybersecurity."
]

distilled_pairs = []

# Step 1: Teacher generates outputs
for text in training_data:
    teacher_response = client.chat.completions.create(
        model=teacher_model,
        messages=[{"role": "user", "content": text}],
        temperature=0.7
    )
    teacher_output = teacher_response.choices[0].message.content
    distilled_pairs.append((text, teacher_output))

# Step 2: Student learns from teacher outputs
for input_text, teacher_output in distilled_pairs:
    student_response = client.chat.completions.create(
        model=student_model,
        messages=[{"role": "user", "content": f"Learn from this teacher answer: {teacher_output}"}],
        temperature=0.7
    )
    print(f"Student learned:\n{student_response.choices[0].message.content}\n")

Student learned:
Overall, cloud computing offers numerous benefits such as cost efficiency, scalability, accessibility, automatic updates, and disaster recovery. However, it also poses challenges including security and privacy concerns, downtime risks, and limited control over infrastructure. Understanding the key characteristics, deployment models, service models, advantages, and challenges of cloud computing is essential for individuals and organizations looking to leverage this technology effectively. By embracing cloud computing, businesses can optimize their operations, improve flexibility, and drive innovation in today's digital landscape.

Student learned:
Additionally, reinforcement learning can handle sequential decision-making tasks and adapt to changing environments. It can also learn from sparse rewards and explore unknown environments effectively. However, RL also faces challenges such as the need for extensive exploration, dealing with high-dimensional state and action sp

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load a smaller distilled model
model_name = "distilbert-base-uncased"  # example distilled model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Apply dynamic quantization (INT8)
quantized_model = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

# Run inference with quantized model
inputs = tokenizer("Explain neural networks simply.", return_tensors="pt")
outputs = quantized_model.generate(**inputs, max_length=50)

print("Quantized Output:", tokenizer.decode(outputs[0], skip_special_tokens=True))


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

D:\AI-Automation\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

ValueError: Unrecognized configuration class <class 'transformers.models.distilbert.configuration_distilbert.DistilBertConfig'> for this kind of AutoModel: AutoModelForCausalLM.
Model type should be one of GPT2Config, AfmoeConfig, ApertusConfig, ArceeConfig, AriaTextConfig, AXK1Config, AXK2Config, BailingHybridConfig, BambaConfig, BartConfig, BeamConfig, BertConfig, BertGenerationConfig, BigBirdConfig, BigBirdPegasusConfig, BioGptConfig, BitNetConfig, BlenderbotConfig, BlenderbotSmallConfig, BloomConfig, BltConfig, CamembertConfig, CodeGenConfig, CohereConfig, Cohere2Config, Cohere2MoeConfig, CohereCompassTextConfig, CpmAntConfig, CTRLConfig, CwmConfig, Data2VecTextConfig, DbrxConfig, DeepseekV2Config, DeepseekV3Config, DeepseekV32Config, DeepseekV4Config, DiffLlamaConfig, DogeConfig, Dots1Config, ElectraConfig, Emu3Config, ErnieConfig, Ernie4_5Config, Ernie4_5_MoeConfig, Exaone4Config, ExaoneMoeConfig, FalconConfig, FalconH1Config, FalconMambaConfig, FlexOlmoConfig, FuyuConfig, GemmaC